In [1]:
import os
import yaml
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm
from huggingface_hub import hf_hub_download

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

# ClinVar

In [2]:
BENCH_DIR = "/s/project/deeprvat/ukb_gym/other_benchmarks"

In [3]:
# Download ClinVar VCF (GRCh38) and its index
!mkdir -p {BENCH_DIR}/clinvar
!wget -nc https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz -O {BENCH_DIR}/clinvar/clinvar.vcf.gz
!wget -nc https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz.tbi -O {BENCH_DIR}/clinvar/clinvar.vcf.gz.tbi

--2026-01-26 17:40:02--  https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 2607:f220:41e:250::31, 2607:f220:41e:250::7, 2607:f220:41e:250::13, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|2607:f220:41e:250::31|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 186228201 (178M) [application/x-gzip]
Saving to: ‘/s/project/deeprvat/ukb_gym/other_benchmarks/clinvar/clinvar.vcf.gz’

/s/project/deeprvat 100%[===================>] 177.60M  15.8MB/s    in 12s     

2026-01-26 17:40:15 (14.5 MB/s) - ‘/s/project/deeprvat/ukb_gym/other_benchmarks/clinvar/clinvar.vcf.gz’ saved [186228201/186228201]

--2026-01-26 17:40:15--  https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz.tbi
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 2607:f220:41e:250::31, 2607:f220:41e:250::7, 2607:f220:41e:250::13, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|2607:f22

In [4]:
# 1. Create a dummy dataframe with your example string (added CLNSIG for demonstration)
data = {
    "info_col": [
        ".11:g.917887G>T;CLNVC=single_nucleotide_variant;CLNVCSO=SO:0001483;GENEINFO=LINC02593:100130417;MC=SO:0001627|intron_variant;ORIGIN=0",
        "CLNVC=single_nucleotide_variant;CLNSIG=Likely_benign;MC=SO:0001583|missense_variant;ORIGIN=1",
        "CLNVC=deletion;CLNSIG=Pathogenic;MC=SO:0001587|stop_gained;ORIGIN=1"
    ]
}
df = pl.DataFrame(data)

# 2. Extract the fields using Regex
# We create two new columns by parsing the 'info_col'
df.with_columns(
    # Regex explanation for MC: 
    # Look for "MC=", skip characters until pipe "|", capture text until next comma or semicolon
    variant_region = pl.col("info_col").str.extract(r"MC=[^|]+\|([^;,]+)", 1),
    
    # Regex explanation for CLNSIG: 
    # Look for "CLNSIG=", capture everything until the next semicolon
    clinical_significance = pl.col("info_col").str.extract(r"CLNSIG=([^;]+)", 1)
)

info_col,variant_region,clinical_significance
str,str,str
""".11:g.917887G>T;CLNVC=single_n…","""intron_variant""",null
"""CLNVC=single_nucleotide_varian…","""missense_variant""","""Likely_benign"""
"""CLNVC=deletion;CLNSIG=Pathogen…","""stop_gained""","""Pathogenic"""


In [23]:
clinvar = (
    pl.read_csv(f"{BENCH_DIR}/clinvar/clinvar.vcf.gz", comment_prefix="##", separator="\t", ignore_errors=True)
    .with_columns(
        chrom = pl.col("#CHROM").fill_null("NA"),
    
        # Look for "CLNSIG=", capture everything until the next semicolon
        clinical_significance = pl.col("INFO").str.extract(r"CLNSIG=([^;]+)", 1),
    )
    .with_columns(
        id = pl.col("chrom").cast(str) + ":" + pl.col("POS").cast(str) + ":" + pl.col("REF") + ":" + pl.col("ALT"),

        # Look for "MC=", skip characters until pipe "|", capture text until next comma or semicolon
        variant_region = pl.col("INFO").str.extract(r"MC=[^|]+\|([^;,]+)", 1),

        pathogenic = (
            pl.when(pl.col("clinical_significance").is_in(["Pathogenic", "Likely_pathogenic"])).then(1)
            .otherwise(pl.when(pl.col("clinical_significance").is_in(["Benign", "Likely_benign"])).then(0).otherwise(None))
        ),

        benchmark = pl.lit("ClinVar"),
    )
)

clinvar

#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,chrom,clinical_significance,id,variant_region,pathogenic,benchmark
i64,i64,i64,str,str,str,str,str,str,str,str,str,i32,str
1,66926,3385321,"""AG""","""A""",""".""",""".""","""ALLELEID=3544463;CLNDISDB=Huma…","""1""","""Uncertain_significance""","""1:66926:AG:A""","""intron_variant""",null,"""ClinVar"""
1,69134,2205837,"""A""","""G""",""".""",""".""","""ALLELEID=2193183;CLNDISDB=MedG…","""1""","""Likely_benign""","""1:69134:A:G""","""missense_variant""",0,"""ClinVar"""
1,69241,4562067,"""C""","""T""",""".""",""".""","""ALLELEID=4679177;CLNDISDB=MedG…","""1""","""Uncertain_significance""","""1:69241:C:T""","""missense_variant""",null,"""ClinVar"""
1,69308,3925305,"""A""","""G""",""".""",""".""","""ALLELEID=4039319;CLNDISDB=MedG…","""1""","""Uncertain_significance""","""1:69308:A:G""","""missense_variant""",null,"""ClinVar"""
1,69314,3205580,"""T""","""G""",""".""",""".""","""ALLELEID=3374047;CLNDISDB=MedG…","""1""","""Uncertain_significance""","""1:69314:T:G""","""missense_variant""",null,"""ClinVar"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
null,274185,3778023,"""C""","""T""",""".""",""".""","""ALLELEID=3894028;CLNDISDB=MedG…","""NA""","""Likely_benign""","""NA:274185:C:T""","""intron_variant""",0,"""ClinVar"""
null,274366,2206666,"""G""","""C""",""".""",""".""","""ALLELEID=2200058;CLNDISDB=MedG…","""NA""","""Uncertain_significance""","""NA:274366:G:C""","""missense_variant""",null,"""ClinVar"""
null,275068,2241971,"""T""","""C""",""".""",""".""","""ALLELEID=2226217;CLNDISDB=MedG…","""NA""","""Uncertain_significance""","""NA:275068:T:C""","""missense_variant""",null,"""ClinVar"""


# TraitGym

In [63]:
# Download the file to your local cache and get the path
file_path = hf_hub_download(
    repo_id="songlab/TraitGym", 
    filename="mendelian_traits_matched_9/test.parquet",
    repo_type="dataset"
)

# 1. Define the mapping dictionary
ccre_to_vep = {
    # Promoters
    "PLS": "promoter_variant",
    
    # Enhancers (Proximal and Distal)
    "pELS": "enhancer_variant",
    "dELS": "enhancer_variant",
    "dELS_flank": "regulatory_region_variant", # Generic regulatory term for flanking regions
    
    # Transcription Factors
    "CTCF-only": "TF_binding_site_variant",
    
    # Ambiguous / Poised regions
    "DNase-H3K4me3": "regulatory_region_variant" 
}


# Read the local file
trait_gym_m = (
    pl.read_parquet(file_path)
    .with_columns(
        id = pl.concat_str([
            pl.col("chrom").cast(pl.Utf8),
            pl.lit(":"),
            pl.col("pos").cast(pl.Utf8),
            pl.lit(":"),
            pl.col("ref"),
            pl.lit(":"),
            pl.col("alt")
        ]),
        variant_region = (
            pl.when(pl.col("consequence").is_in(ccre_to_vep.keys()))
            .then(pl.col("consequence").replace(ccre_to_vep, default=None))
            .otherwise(pl.col("consequence"))
        ),
        pathogenic = pl.col('label'),
        benchmark = pl.lit("TraitGym mendelian traits (matched)")
    )
)
trait_gym_m

/scratch/tmp/lond/ipykernel_101434/837040322.py:41: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)


chrom,pos,ref,alt,OMIM,consequence,label,tss_dist,match_group,id,variant_region,pathogenic,benchmark
str,i64,str,str,str,str,bool,i64,str,str,str,bool,str
"""1""",1425822,"""C""","""G""",null,"""PLS""",false,48,"""PLS_4""","""1:1425822:C:G""","""promoter_variant""",false,"""TraitGym mendelian traits (mat…"
"""1""",1615869,"""C""","""T""",null,"""PLS""",false,35,"""PLS_0""","""1:1615869:C:T""","""promoter_variant""",false,"""TraitGym mendelian traits (mat…"
"""1""",1659060,"""G""","""A""",null,"""PLS""",false,47,"""PLS_4""","""1:1659060:G:A""","""promoter_variant""",false,"""TraitGym mendelian traits (mat…"
"""1""",1659114,"""A""","""G""",null,"""PLS""",false,101,"""PLS_5""","""1:1659114:A:G""","""promoter_variant""",false,"""TraitGym mendelian traits (mat…"
"""1""",2050958,"""T""","""C""",null,"""5_prime_UTR_variant""",false,149,"""5_prime_UTR_variant_7""","""1:2050958:T:C""","""5_prime_UTR_variant""",false,"""TraitGym mendelian traits (mat…"
…,…,…,…,…,…,…,…,…,…,…,…,…
"""X""",155613005,"""C""","""T""",null,"""PLS""",false,52,"""PLS_52""","""X:155613005:C:T""","""promoter_variant""",false,"""TraitGym mendelian traits (mat…"
"""X""",155719093,"""C""","""A""",null,"""5_prime_UTR_variant""",false,4,"""5_prime_UTR_variant_101""","""X:155719093:C:A""","""5_prime_UTR_variant""",false,"""TraitGym mendelian traits (mat…"
"""X""",155881342,"""A""","""C""",null,"""PLS""",false,2,"""PLS_57""","""X:155881342:A:C""","""promoter_variant""",false,"""TraitGym mendelian traits (mat…"


In [65]:
# Download the file to your local cache and get the path
file_path = hf_hub_download(
    repo_id="songlab/TraitGym", 
    filename="complex_traits_matched_9/test.parquet",
    repo_type="dataset"
)

# Read the local file
trait_gym_c = (
    pl.read_parquet(file_path)
    .with_columns(
        id = pl.concat_str([
            pl.col("chrom").cast(pl.Utf8),
            pl.lit(":"),
            pl.col("pos").cast(pl.Utf8),
            pl.lit(":"),
            pl.col("ref"),
            pl.lit(":"),
            pl.col("alt")
        ]),
        variant_region = (
            pl.when(pl.col("consequence").is_in(ccre_to_vep.keys()))
            .then(pl.col("consequence").replace(ccre_to_vep, default=None))
            .otherwise(pl.col("consequence"))
        ),
        pathogenic = pl.col('label'),
        benchmark = pl.lit("TraitGym mendelian traits (matched)")
    )
)
trait_gym_c

/scratch/tmp/lond/ipykernel_101434/3739900729.py:23: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)


chrom,pos,ref,alt,pip,trait,label,maf,ld_score,consequence,tss_dist,match_group,id,variant_region,pathogenic,benchmark
str,i64,str,str,f64,str,bool,f64,f64,str,i64,str,str,str,bool,str
"""1""",867476,"""C""","""T""",0.00156,"""""",false,0.079465,44.053,"""non_coding_transcript_exon_var…",56446,"""non_coding_transcript_exon_var…","""1:867476:C:T""","""non_coding_transcript_exon_var…",false,"""TraitGym mendelian traits (mat…"
"""1""",868052,"""T""","""C""",0.001791,"""""",false,0.077747,44.057,"""non_coding_transcript_exon_var…",55870,"""non_coding_transcript_exon_var…","""1:868052:T:C""","""non_coding_transcript_exon_var…",false,"""TraitGym mendelian traits (mat…"
"""1""",868635,"""A""","""G""",0.004349,"""""",false,0.075255,43.639,"""non_coding_transcript_exon_var…",55287,"""non_coding_transcript_exon_var…","""1:868635:A:G""","""non_coding_transcript_exon_var…",false,"""TraitGym mendelian traits (mat…"
"""1""",870176,"""T""","""A""",0.0,"""""",false,0.084371,37.271,"""non_coding_transcript_exon_var…",53746,"""non_coding_transcript_exon_var…","""1:870176:T:A""","""non_coding_transcript_exon_var…",false,"""TraitGym mendelian traits (mat…"
"""1""",1052930,"""A""","""G""",0.001467,"""""",false,0.058385,46.907,"""non_coding_transcript_exon_var…",18823,"""non_coding_transcript_exon_var…","""1:1052930:A:G""","""non_coding_transcript_exon_var…",false,"""TraitGym mendelian traits (mat…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""22""",50368376,"""T""","""C""",0.0,"""""",false,0.19181,86.507,"""dELS""",3695,"""dELS_204""","""22:50368376:T:C""","""enhancer_variant""",false,"""TraitGym mendelian traits (mat…"
"""22""",50571623,"""C""","""T""",0.0,"""""",false,0.061159,23.9,"""dELS""",6291,"""dELS_202""","""22:50571623:C:T""","""enhancer_variant""",false,"""TraitGym mendelian traits (mat…"
"""22""",50671289,"""G""","""A""",0.0,"""""",false,0.036223,12.733,"""pELS_flank""",3125,"""pELS_flank_26""","""22:50671289:G:A""","""pELS_flank""",false,"""TraitGym mendelian traits (mat…"


# ProteinGym

In [37]:
VERSION = "v1.3"
FILENAME = "DMS_ProteinGym_substitutions.zip"
TARGET_DIR = f"{BENCH_DIR}/protein_gym"
TARGET_FILE = f"{TARGET_DIR}/{FILENAME}"

# 1. Create Directory
!mkdir -p {TARGET_DIR}

# 2. Download only if missing
if not os.path.exists(TARGET_FILE):
    print("File not found. Downloading...")
    !curl -o {TARGET_FILE} https://marks.hms.harvard.edu/proteingym/ProteinGym_{VERSION}/{FILENAME}
else:
    print("File already exists. Skipping download.")

# 3. Unzip with "Yes to All" (-o)
# We use -o to overwrite if it exists, avoiding the interactive prompt
print("Unzipping...")
!unzip -o {TARGET_FILE} -d {TARGET_DIR}

File already exists. Skipping download.
Unzipping...
Archive:  /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions.zip
  inflating: /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions/SDA_BACSU_Tsuboyama_2023_1PV0.csv  
  inflating: /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions/PAI1_HUMAN_Huttinger_2021.csv  
  inflating: /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions/S22A1_HUMAN_Yee_2023_activity.csv  
  inflating: /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions/HIS7_YEAST_Pokusaeva_2019.csv  
  inflating: /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions/AMIE_PSEAE_Wrenbeck_2017.csv  
  inflating: /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions/ACE2_HUMAN_Chan_2020.csv  
  inflating: /s/project/deeprvat/ukb_gym/other_benchmarks/p

In [28]:
VERSION = "v1.3"
FILENAME = "DMS_ProteinGym_indels.zip"
TARGET_DIR = f"{BENCH_DIR}/protein_gym"
TARGET_FILE = f"{TARGET_DIR}/{FILENAME}"

# 1. Create Directory
!mkdir -p {TARGET_DIR}

# 2. Download only if missing
if not os.path.exists(TARGET_FILE):
    print("File not found. Downloading...")
    !curl -o {TARGET_FILE} https://marks.hms.harvard.edu/proteingym/ProteinGym_{VERSION}/{FILENAME}
else:
    print("File already exists. Skipping download.")

# 3. Unzip with "Yes to All" (-o)
# We use -o to overwrite if it exists, avoiding the interactive prompt
print("Unzipping...")
!unzip -o {TARGET_FILE} -d {TARGET_DIR}

File not found. Downloading...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 6552k  100 6552k    0     0  5305k      0  0:00:01  0:00:01 --:--:-- 5305k 567k      0  0:00:11 --:--:--  0:00:11  567k
Unzipping...
Archive:  /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_ProteinGym_indels.zip
   creating: /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_ProteinGym_indels/
  inflating: /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_ProteinGym_indels/SDA_BACSU_Tsuboyama_2023_1PV0_indels.csv  
  inflating: /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_ProteinGym_indels/PR40A_HUMAN_Tsuboyama_2023_1UZC_indels.csv  
  inflating: /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_ProteinGym_indels/BBC1_YEAST_Tsuboyama_2023_1TG0_indels.csv  
  inflating: /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_Protei

In [ ]:
# csv_pattern = f"{TARGET_DIR}/**/*HUMAN*.csv"
csv_pattern = f"{TARGET_DIR}/DMS_ProteinGym_substitutions/*HUMAN*.csv"

print(f"Scanning files matching: {csv_pattern}")

# 2. Scan and Compile
# 'include_file_paths' creates a column with the file path, which is 
# critical to distinguish which assay the variants come from.
proteingym_snp = (
    pl.scan_csv(csv_pattern, include_file_paths="source_file")
    .with_columns(
        # Extract just the assay name (remove path and extension)
        assay_id = pl.col("source_file")
                    .str.split("/")
                    .list.last()
                    .str.strip_suffix(".csv")
    )
    .collect()

    .with_columns(
        # 1. Flag Bottom 1%
        is_bottom_1pct = pl.col("DMS_score") <= pl.col("DMS_score").quantile(0.01).over("assay_id"),
        
        # 2. Flag Top 1%
        is_top_1pct = pl.col("DMS_score") >= pl.col("DMS_score").quantile(0.99).over("assay_id")
    )
    .with_columns(
        id = pl.col('mutant'),
        variant_region = pl.lit("missense_variant"),
        pathogenic = pl.col("is_bottom_1pct") | pl.col("is_top_1pct"),
        benchmark = pl.lit("ProteinGym SNP (Human)"),
    )
)

proteingym_snp

Scanning files matching: /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions/*HUMAN*.csv


mutant,mutated_sequence,DMS_score,DMS_score_bin,source_file,assay_id,is_bottom_1pct,is_top_1pct,id,variant_region,pathogenic,benchmark
str,str,f64,i64,str,str,bool,bool,str,str,bool,str
"""A673C""","""MLPGLALLLLAAWTARALEVPTDGNAGLLA…",-1.018869,1,"""/s/project/deeprvat/ukb_gym/ot…","""A4_HUMAN_Seuma_2022""",false,false,"""A673C""","""missense_variant""",false,"""ProteinGym SNP (human)"""
"""A673D""","""MLPGLALLLLAAWTARALEVPTDGNAGLLA…",-0.605052,1,"""/s/project/deeprvat/ukb_gym/ot…","""A4_HUMAN_Seuma_2022""",false,false,"""A673D""","""missense_variant""",false,"""ProteinGym SNP (human)"""
"""A673E""","""MLPGLALLLLAAWTARALEVPTDGNAGLLA…",-0.590857,1,"""/s/project/deeprvat/ukb_gym/ot…","""A4_HUMAN_Seuma_2022""",false,false,"""A673E""","""missense_variant""",false,"""ProteinGym SNP (human)"""
"""A673E:A692E""","""MLPGLALLLLAAWTARALEVPTDGNAGLLA…",-2.443601,0,"""/s/project/deeprvat/ukb_gym/ot…","""A4_HUMAN_Seuma_2022""",false,false,"""A673E:A692E""","""missense_variant""",false,"""ProteinGym SNP (human)"""
"""A673E:A692T""","""MLPGLALLLLAAWTARALEVPTDGNAGLLA…",-3.049893,0,"""/s/project/deeprvat/ukb_gym/ot…","""A4_HUMAN_Seuma_2022""",false,false,"""A673E:A692T""","""missense_variant""",false,"""ProteinGym SNP (human)"""
…,…,…,…,…,…,…,…,…,…,…,…
"""R203H""","""MDPGQQPPPQPAPQGQGQPPSQPPQGQGPP…",0.124027,0,"""/s/project/deeprvat/ukb_gym/ot…","""YAP1_HUMAN_Araya_2012""",false,false,"""R203H""","""missense_variant""",false,"""ProteinGym SNP (human)"""
"""R203G""","""MDPGQQPPPQPAPQGQGQPPSQPPQGQGPP…",0.150523,0,"""/s/project/deeprvat/ukb_gym/ot…","""YAP1_HUMAN_Araya_2012""",false,false,"""R203G""","""missense_variant""",false,"""ProteinGym SNP (human)"""
"""R203D""","""MDPGQQPPPQPAPQGQGQPPSQPPQGQGPP…",0.138661,0,"""/s/project/deeprvat/ukb_gym/ot…","""YAP1_HUMAN_Araya_2012""",false,false,"""R203D""","""missense_variant""",false,"""ProteinGym SNP (human)"""


In [ ]:
# csv_pattern = f"{TARGET_DIR}/**/*HUMAN*.csv"
csv_pattern = f"{TARGET_DIR}/DMS_ProteinGym_indels/*HUMAN*.csv"

print(f"Scanning files matching: {csv_pattern}")

# 2. Scan and Compile
# 'include_file_paths' creates a column with the file path, which is 
# critical to distinguish which assay the variants come from.
proteingym_indel = (
    pl.scan_csv(csv_pattern, include_file_paths="source_file")
    .with_columns(
        # Extract just the assay name (remove path and extension)
        assay_id = pl.col("source_file")
                    .str.split("/")
                    .list.last()
                    .str.strip_suffix(".csv")
    )
    .collect()

    .with_columns(
        # 1. Flag Bottom 1%
        is_bottom_1pct = pl.col("DMS_score") <= pl.col("DMS_score").quantile(0.01).over("assay_id"),
        
        # 2. Flag Top 1%
        is_top_1pct = pl.col("DMS_score") >= pl.col("DMS_score").quantile(0.99).over("assay_id")
    )
    .with_columns(
        id = pl.col('mutated_sequence'),
        variant_region = pl.lit("frameshift_variant"),
        pathogenic = pl.col("is_bottom_1pct") | pl.col("is_top_1pct"),
        benchmark = pl.lit("ProteinGym indel (Human)")
    )
)

proteingym_indel

Scanning files matching: /s/project/deeprvat/ukb_gym/other_benchmarks/protein_gym/DMS_ProteinGym_indels/*HUMAN*.csv


mutated_sequence,DMS_score,DMS_score_bin,source_file,assay_id,is_bottom_1pct,is_top_1pct,id,variant_region,pathogenic,benchmark
str,f64,i64,str,str,bool,bool,str,str,bool,str
"""MLPGLALLLLAAWTARALEVPTDGNAGLLA…",-3.311444,0,"""/s/project/deeprvat/ukb_gym/ot…","""A4_HUMAN_Seuma_2022_indels""",false,false,"""MLPGLALLLLAAWTARALEVPTDGNAGLLA…","""frameshift_variant""",false,"""ProteinGym indel (human)"""
"""MLPGLALLLLAAWTARALEVPTDGNAGLLA…",-3.36364,0,"""/s/project/deeprvat/ukb_gym/ot…","""A4_HUMAN_Seuma_2022_indels""",false,false,"""MLPGLALLLLAAWTARALEVPTDGNAGLLA…","""frameshift_variant""",false,"""ProteinGym indel (human)"""
"""MLPGLALLLLAAWTARALEVPTDGNAGLLA…",-3.413797,0,"""/s/project/deeprvat/ukb_gym/ot…","""A4_HUMAN_Seuma_2022_indels""",false,false,"""MLPGLALLLLAAWTARALEVPTDGNAGLLA…","""frameshift_variant""",false,"""ProteinGym indel (human)"""
"""MLPGLALLLLAAWTARALEVPTDGNAGLLA…",-3.352993,0,"""/s/project/deeprvat/ukb_gym/ot…","""A4_HUMAN_Seuma_2022_indels""",false,false,"""MLPGLALLLLAAWTARALEVPTDGNAGLLA…","""frameshift_variant""",false,"""ProteinGym indel (human)"""
"""MLPGLALLLLAAWTARALEVPTDGNAGLLA…",0.235065,1,"""/s/project/deeprvat/ukb_gym/ot…","""A4_HUMAN_Seuma_2022_indels""",false,false,"""MLPGLALLLLAAWTARALEVPTDGNAGLLA…","""frameshift_variant""",false,"""ProteinGym indel (human)"""
…,…,…,…,…,…,…,…,…,…,…
"""HRQALGGERLYPRVQAMQPAFASKITGMLL…",-1.300614,0,"""/s/project/deeprvat/ukb_gym/ot…","""UBR5_HUMAN_Tsuboyama_2023_1I2T…",false,false,"""HRQALGGERLYPRVQAMQPAFASKITGMLL…","""frameshift_variant""",false,"""ProteinGym indel (human)"""
"""HRQALGRLYPRVQAMQPAFASKITGMLLEL…",-0.619357,0,"""/s/project/deeprvat/ukb_gym/ot…","""UBR5_HUMAN_Tsuboyama_2023_1I2T…",false,false,"""HRQALGRLYPRVQAMQPAFASKITGMLLEL…","""frameshift_variant""",false,"""ProteinGym indel (human)"""
"""HRQGALGERLYPRVQAMQPAFASKITGMLL…",-0.421069,1,"""/s/project/deeprvat/ukb_gym/ot…","""UBR5_HUMAN_Tsuboyama_2023_1I2T…",false,false,"""HRQGALGERLYPRVQAMQPAFASKITGMLL…","""frameshift_variant""",false,"""ProteinGym indel (human)"""


# GeneticGym

In [46]:
!mkdir -p {BENCH_DIR}/genetics_gym
!wget -nc https://storage.googleapis.com/genetics-gym/evaluation_tables/dd-with-combined-controls.tsv.bgz -O {BENCH_DIR}/genetics_gym/dd-with-combined-controls.tsv.bgz
!wget -nc https://storage.googleapis.com/genetics-gym/evaluation_tables/asd-with-combined-controls.tsv.bgz -O {BENCH_DIR}/genetics_gym/asd-with-combined-controls.tsv.bgz
!wget -nc https://storage.googleapis.com/genetics-gym/evaluation_tables/chd-with-combined-controls.tsv.bgz -O {BENCH_DIR}/genetics_gym/chd-with-combined-controls.tsv.bgz

!wget -nc https://storage.googleapis.com/genetics-gym/evaluation_tables/schema_evaluation_table.tsv.bgz -O {BENCH_DIR}/genetics_gym/schema_evaluation_table.tsv.bgz
!wget -nc https://storage.googleapis.com/genetics-gym/evaluation_tables/asc_evaluation_table.tsv.bgz -O {BENCH_DIR}/genetics_gym/asc_evaluation_table.tsv.bgz
!wget -nc https://storage.googleapis.com/genetics-gym/evaluation_tables/epi25_evaluation_table.tsv.bgz -O {BENCH_DIR}/genetics_gym/epi25_evaluation_table.tsv.bgz


!wget -nc https://storage.googleapis.com/genetics-gym/evaluation_tables/genebass_evaluation_table.tsv.bgz -O {BENCH_DIR}/genetics_gym/genebass_evaluation_table.tsv.bgz
!wget -nc https://storage.googleapis.com/genetics-gym/evaluation_tables/gnomad-independent-set.tsv.bgz -O {BENCH_DIR}/genetics_gym/gnomad-independent-set.tsv.bgz

File ‘/s/project/deeprvat/ukb_gym/other_benchmarks/genetics_gym/dd-with-combined-controls.tsv.bgz’ already there; not retrieving.
--2026-01-26 18:35:57--  https://storage.googleapis.com/genetics-gym/evaluation_tables/asd-with-combined-controls.tsv.bgz
Resolving storage.googleapis.com (storage.googleapis.com)... 2a00:1450:4001:804::201b, 2a00:1450:4001:80d::201b, 2a00:1450:4001:831::201b, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|2a00:1450:4001:804::201b|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 95112 (93K) [application/octet-stream]
Saving to: ‘/s/project/deeprvat/ukb_gym/other_benchmarks/genetics_gym/asd-with-combined-controls.tsv.bgz’

/s/project/deeprvat 100%[===================>]  92.88K   580KB/s    in 0.2s    

2026-01-26 18:35:58 (580 KB/s) - ‘/s/project/deeprvat/ukb_gym/other_benchmarks/genetics_gym/asd-with-combined-controls.tsv.bgz’ saved [95112/95112]

--2026-01-26 18:35:58--  https://storage.googleapis.com/genetics-gym

In [44]:
geneticsgym_ddd = (
    pl.read_csv(f"{BENCH_DIR}/genetics_gym/dd-with-combined-controls.tsv.bgz", separator="\t")
    
)
geneticsgym_ddd

chrom,pos,ref,alt,is_pos_dd
str,i64,str,str,bool
"""chr1""",935836,"""G""","""A""",true
"""chr1""",942482,"""T""","""C""",true
"""chr1""",942642,"""G""","""A""",true
"""chr1""",945120,"""G""","""C""",true
"""chr1""",948519,"""T""","""C""",true
…,…,…,…,…
"""chrX""",154773227,"""G""","""A""",true
"""chrX""",154776317,"""G""","""A""",true
"""chrX""",154906457,"""C""","""T""",false


In [47]:
geneticsgym = pl.read_csv(f"{BENCH_DIR}/genetics_gym/gnomad-independent-set.tsv.bgz", separator="\t")
geneticsgym

chrom,pos,ref,alt,is_pos
str,i64,str,str,bool
"""chr1""",65431,"""C""","""A""",true
"""chr1""",65431,"""C""","""G""",true
"""chr1""",65431,"""C""","""T""",true
"""chr1""",65432,"""A""","""C""",true
"""chr1""",65432,"""A""","""G""",true
…,…,…,…,…
"""chrX""",156010408,"""A""","""C""",true
"""chrX""",156010408,"""A""","""G""",true
"""chrX""",156010408,"""A""","""T""",true


# Compile all datasets

In [67]:
common_cols = ['id', 'variant_region', 'pathogenic', 'benchmark']

oth_bmrk = pl.concat([
        clinvar.select(common_cols), #.drop_nulls(subset=['pathogenic']),
        trait_gym_m.select(common_cols),
        trait_gym_c.select(common_cols),
        proteingym_snp.select(common_cols),
        proteingym_indel.select(common_cols),
    ],
    how="vertical_relaxed"
)

oth_bmrk_plot = (
    oth_bmrk
    .with_columns(
        n_variants = pl.len().over("benchmark"),
        n_pathogenic = pl.col("pathogenic").cast(pl.Int64).sum().over("benchmark")
    )
    .with_columns(
        n_variants_region = pl.len().over(["benchmark", "variant_region"]),
        n_pathogenic_region = pl.col("pathogenic").cast(pl.Int64).sum().over(["benchmark", "variant_region"])
    )
    .with_columns(
        frac_pathogenic = pl.col("n_pathogenic") / pl.col("n_variants"),
        frac_pathogenic_region = pl.col("n_pathogenic_region") / pl.col("n_variants_region"),
    )
    .filter(pl.col("pathogenic").is_not_null())
    .select([
        "benchmark",
        "variant_region",
        "n_variants",
        "n_pathogenic",
        "n_variants_region",
        "n_pathogenic_region",
        "frac_pathogenic",
        "frac_pathogenic_region"
    ])
    .unique()
)

oth_bmrk_plot

benchmark,variant_region,n_variants,n_pathogenic,n_variants_region,n_pathogenic_region,frac_pathogenic,frac_pathogenic_region
str,str,u64,i64,u64,i64,f64,f64
"""ClinVar""","""genic_upstream_transcript_vari…",4280453,286303,863,54,0.066886,0.062572
"""ClinVar""","""3_prime_UTR_variant""",4280453,286303,72266,275,0.066886,0.003805
"""TraitGym mendelian traits (mat…","""PLS_flank""",14780,1478,170,17,0.1,0.1
"""ProteinGym SNP (human)""","""missense_variant""",485830,12821,485830,12821,0.02639,0.02639
"""ClinVar""",null,4280453,286303,19909,1783,0.066886,0.089557
…,…,…,…,…,…,…,…
"""ClinVar""","""stop_lost""",4280453,286303,1642,273,0.066886,0.166261
"""ClinVar""","""no_sequence_alteration""",4280453,286303,595,2,0.066886,0.003361
"""ClinVar""","""intron_variant""",4280453,286303,654752,3781,0.066886,0.005775


In [ ]:
(
    
)